<a href="https://colab.research.google.com/github/brindha22001-wq/Brindha-D/blob/main/Copy_of_GRU_ipynb_GRU_IMPLEMENTATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gensim # semanic modelling, similar words have similar vectors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.5 MB/s eta 0:00:00


In [ ]:
# Import Required Libraries

import warnings
warnings.filterwarnings("ignore")
import re
import numpy as np
import nltk
nltk.download('punkt')
nltk.download("punkt_tab")

from nltk.tokenize import word_tokenize
import requests # fetch data from the web

from gensim.models import KeyedVectors

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GRU, Embedding


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
# Load Dataset (Ramayana)

url = "https://www.gutenberg.org/files/24869/24869-0.txt"
book = requests.get(url)
data = book.text

# Downloads the Ramayana text from Project Gutenberg.
# Stores the raw text in a string variable 'data'.

In [ ]:
#type(data)
#len(data)#entire book
#print(data[:5000])

In [ ]:
# Extract Main Content (Ignore Metadata)

import re
start_index = re.search("invocation.\(1\)", data, re.I)
data = data[start_index.start():]

# Finds the starting point of the actual story (ignoring metadata).
# Keeps only the relevant story content for training.

In [ ]:
# Define Text Cleaning Function

def clean_document(document, char_filter=r"[^\w]"):
    document = document.lower()
    words = word_tokenize(document)
    document = " ".join(words)
    document = re.sub(char_filter, " ", document)
    document = re.sub(r"\s+", " ", document).strip()
    return document

# Clean text
data = clean_document(data)

# Converts text into a standardized clean format (lowercasing, removing symbols).
# Ensures dataset is consistent for tokenization and training.
# standardizes raw text by lowercasing, removing symbols, and fixing spacing to prepare it for GRU training.

In [ ]:
# Tokenization and Encoding

word_tokeniser = Tokenizer()
word_tokeniser.fit_on_texts([data])#Assigns a unique integer ID to each word based on frequency,Most frequent word, smallest integer
encoded_words = word_tokeniser.texts_to_sequences([data])[0]

VOCABULARY_SIZE = len(word_tokeniser.word_index) + 1 #because Keras reserves index 0 for padding
print('Vocabulary Size:', VOCABULARY_SIZE)

# Builds a vocabulary of all words from the dataset.
# Converts words into unique integer IDs for neural network input.

Vocabulary Size: 17667


In [ ]:
#len(encoded_words )
#encoded_words[:500]

In [ ]:
# Create Training Sequences

MAX_SEQ_LENGTH = 5 # We want the model to look at 5 words and predict the 6th
sequences = []
for i in range(MAX_SEQ_LENGTH, len(encoded_words)):
    seq = encoded_words[i-MAX_SEQ_LENGTH:i+1]
    sequences.append(seq)

import numpy as np
sequences = np.array(sequences)
print('Total training samples:', len(sequences))

# Creates training samples where 5 words - predict 6th word.
# Prepares input-output pairs required for supervised learning.
# Input = previous 5 words

# Output = next word

Total training samples: 410241


In [ ]:
# Split Input (X) and Output (y)

X, y = sequences[:, :-1], sequences[:, -1]
print("Input shape:", X.shape)
print("Output shape:", y.shape)

# Splits each sequence into X (input words) and y (target word).
# X holds context, y holds the next word to predict.
# 50000 → number of training samples

## 5 → length of input sequence (context)

# Output is a single word index for prediction

Input shape: (410241, 5)
Output shape: (410241,)


In [ ]:
# Pad Input Sequences

from tensorflow.keras.preprocessing.sequence import pad_sequences
X = pad_sequences(X, maxlen=MAX_SEQ_LENGTH, padding='pre')

# Ensures all input sequences have the same length.
# Required for feeding into RNN-based models like GRU.
#padding='pre'  adds zeros at the beginning if the sequence is shorter than max length

#Alternative: padding='post' adds zeros at the end

In [ ]:
# Build GRU Model with Keras Embedding

EMBEDDING_SIZE = 100

model = Sequential()
model.add(Embedding(VOCABULARY_SIZE, EMBEDDING_SIZE, input_length=MAX_SEQ_LENGTH)) #Converts word IDs → dense vectors
model.add(GRU(128, return_sequences=True)) # Learn sequential patterns, outputs sequence
model.add(GRU(128))#Summarizes sequence, outputs final state
model.add(Dense(VOCABULARY_SIZE, activation='softmax')) #Predicts next word probability over vocabulary

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])#Loss: sparse_categorical_crossentropy

#Works with integer labels (no need for one-hot encoding)
model.summary()

# Embedding layer turns word IDs into dense vectors.
# GRU layers learn sequence patterns, and Dense predicts next word.

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train Model
model.fit(X, y, epochs=10, batch_size=256, verbose=2)

# Trains the GRU model for 10 epochs using mini-batches of 256.
# The model learns to predict the next word from a sequence of 5 words.

Epoch 1/10
1603/1603 - 27s - 17ms/step - accuracy: 0.0756 - loss: 6.8458
Epoch 2/10
1603/1603 - 38s - 23ms/step - accuracy: 0.1076 - loss: 6.0985
Epoch 3/10
1603/1603 - 20s - 13ms/step - accuracy: 0.1243 - loss: 5.7212
Epoch 4/10
1603/1603 - 20s - 13ms/step - accuracy: 0.1353 - loss: 5.4582
Epoch 5/10
1603/1603 - 21s - 13ms/step - accuracy: 0.1441 - loss: 5.2464
Epoch 6/10
1603/1603 - 20s - 13ms/step - accuracy: 0.1532 - loss: 5.0612
Epoch 7/10
1603/1603 - 20s - 13ms/step - accuracy: 0.1618 - loss: 4.8927
Epoch 8/10
1603/1603 - 21s - 13ms/step - accuracy: 0.1722 - loss: 4.7367
Epoch 9/10
1603/1603 - 21s - 13ms/step - accuracy: 0.1840 - loss: 4.5915
Epoch 10/10
1603/1603 - 20s - 13ms/step - accuracy: 0.1976 - loss: 4.4543


In [ ]:
# Text Generation Function

def generate_words(model, word_tokeniser, MAX_SEQ_LENGTH, seed, n_words):
    text = seed #Starts with the given seed phrase

#Generated words will be appended to this
    for _ in range(n_words):#Repeats prediction n_words times
        encoded = word_tokeniser.texts_to_sequences([text])[0]
        padded = pad_sequences([encoded], maxlen=MAX_SEQ_LENGTH, padding='pre')
        pred = model.predict(padded, verbose=0)
        next_index = np.argmax(pred, axis=-1)[0] #Selects word with highest probability
        for word, idx in word_tokeniser.word_index.items(): #Finds the actual word corresponding to the predicted index
            if idx == next_index:#Reverse lookup in vocabulary
                next_word = word
                break
        text += " " + next_word
    return text

# Takes a seed phrase and predicts the next word repeatedly.
# Builds a generated sequence word-by-word using the trained GRU.

In [ ]:
# Using the first model (without pretrained embeddings)

seed_text = "rama never told anyone about"
print(generate_words(model, word_tokeniser, MAX_SEQ_LENGTH, seed_text, 10))

rama never told anyone about the sea and the sun is the sun is the


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

encoded_words = tokenizer.texts_to_sequences([data])[0]
VOCABULARY_SIZE = len(tokenizer.word_index) + 1

In [ ]:
import gensim.downloader as api

EMBEDDING_SIZE = 100

# Automatically downloads glove-wiki-gigaword-100, word2vec
glove_model = api.load("glove-wiki-gigaword-100")

embedding_matrix = np.zeros((VOCABULARY_SIZE, EMBEDDING_SIZE))

for word, index in tokenizer.word_index.items():
    if word in glove_model:
        embedding_matrix[index] = glove_model[word]

[==================================================] 100.0% 128.1/128.1MB downloaded


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense

model = Sequential()
model.add(
    Embedding(
        input_dim=VOCABULARY_SIZE,
        output_dim=EMBEDDING_SIZE,
        weights=[embedding_matrix],   #  use GloVe here
        input_length=MAX_SEQ_LENGTH,
        trainable=False               # freeze pretrained embeddings
    )
)
model.add(GRU(128, return_sequences=True))
model.add(GRU(128))
model.add(Dense(VOCABULARY_SIZE, activation='softmax'))

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │     1,766,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,766,700 (6.74 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 1,766,700 (6.74 MB)

In [ ]:
model.fit(
    X,
    y,
    epochs=10,
    batch_size=128
)

Epoch 1/10
3206/3206 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - accuracy: 0.1246 - loss: 5.5535
Epoch 2/10
3206/3206 ━━━━━━━━━━━━━━━━━━━━ 34s 11ms/step - accuracy: 0.1339 - loss: 5.3017
Epoch 3/10
3206/3206 ━━━━━━━━━━━━━━━━━━━━ 34s 11ms/step - accuracy: 0.1425 - loss: 5.0964
Epoch 4/10
3206/3206 ━━━━━━━━━━━━━━━━━━━━ 33s 10ms/step - accuracy: 0.1500 - loss: 4.9134
Epoch 5/10
3206/3206 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step - accuracy: 0.1598 - loss: 4.7783
Epoch 6/10
3206/3206 ━━━━━━━━━━━━━━━━━━━━ 34s 11ms/step - accuracy: 0.1708 - loss: 4.6452
Epoch 7/10
3206/3206 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - accuracy: 0.1800 - loss: 4.5424
Epoch 8/10
3206/3206 ━━━━━━━━━━━━━━━━━━━━ 34s 11ms/step - accuracy: 0.1898 - loss: 4.4554
Epoch 9/10
3206/3206 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - accuracy: 0.1990 - loss: 4.3634
Epoch 10/10
3206/3206 ━━━━━━━━━━━━━━━━━━━━ 34s 11ms/step - accuracy: 0.2099 - loss: 4.2767


In [ ]:
def generate_words(model, tokenizer, seed_text, n_words):
    text = seed_text

    for _ in range(n_words):
        encoded = tokenizer.texts_to_sequences([text])[0]
        encoded = pad_sequences([encoded], maxlen=MAX_SEQ_LENGTH, padding='pre')

        pred = model.predict(encoded, verbose=0)
        next_index = np.argmax(pred)

        for word, idx in tokenizer.word_index.items():
            if idx == next_index:
                text += " " + word
                break

    return text

In [ ]:
seed_text = "rama never told anyone about"
print(generate_words(model, tokenizer, seed_text, 20))

rama never told anyone about the tale of the sea and the ganges and the indus 935 the poet s book the best of kings
